# desc-wfmon/mondump.ipynb
Display the schema for the monitoring table used in DESC gen3_workflow.  

We assume [desc-wfmon](https://github.com/LSSTDESC/desc-wfmon) has been installed using the install notebook.

In [1]:
%run install/setup.py
import sys
import collections
import pandas
import desc.wfmon
import desc.sysmon

print(f"Python version is {sys.version}")
for pkg in [desc.wfmon, desc.sysmon]:
    print(f"{pkg} version is {pkg.__version__}")

TestMonDbReader:ctor: Done.
Python version is 3.9.7 (default, Sep 16 2021, 13:09:58) 
[GCC 7.5.0]
<module 'desc.wfmon' from '/pscratch/sd/d/dladams/tmprun/ptest70-ifixn-wq-pmcp016-shm07-bt40-021/./install/noconda/desc/wfmon/__init__.py'> version is 0.23.0.dev1
<module 'desc.sysmon' from '/pscratch/sd/d/dladams/tmprun/ptest70-ifixn-wq-pmcp016-shm07-bt40-021/./install/noconda/desc/sysmon/__init__.py'> version is 0.23.0.dev1


## Configuration
List the system and process monitoring files for which we want schema.

In [2]:
# List of files to display
sysfils = ['sysmon.csv']
prcfils = ['runinfo/monitoring.db']


# Set the level for process tables.
lev = 2

# Set units for the memory.
bunit, sbuinit = 1, 'byte'
#bunit, sbunit = 2**20, 'MB'
#bunit, sbunit = 2**30, 'GB'

## Fetch system-level monitoring schema.

In [3]:
line = '----------------------------------------------------------------------------'
print(line)
for sysfil in sysfils:
    if os.path.exists(sysfil):
        print(f"System monitor file: {sysfil}")
        sym = pandas.read_csv(sysfil)
        print(f"System monitor sample count: {len(sym)}")
        print(f"System monitor columns:")
        for cnam in sym.columns:
            print(f"  {cnam}")   
        assert(len(sym.cpu_count.unique()) == 1)
        ncpu = sym.cpu_count[0]
        print(f"CPU count is {ncpu:.0f}")
        assert(len(sym.mem_total.unique()) == 1)
        maxmem = sym.mem_total[0]
        print(f"Total memory is {maxmem:.1f} GB")
    else:
        print(f"File not found: {sysfil}")
    print(line)

----------------------------------------------------------------------------
System monitor file: sysmon.csv
System monitor sample count: 456
System monitor columns:
  time
  cpu_count
  cpu_percent
  cpu_freq
  cpu_user
  cpu_system
  cpu_idle
  cpu_iowait
  cpu_time
  mem_total
  mem_available
  mem_swapfree
  dio_readsize
  dio_writesize
  nio_readsize
  nio_writesize
CPU count is 256
Total memory is 503.2 GB
----------------------------------------------------------------------------


## Fetch the process-level monitoring schema

The process monitoring data is read from the mysql DB produced by parsl. Of particular interest is the task table where metrics are sampled at regular intervals seprately for each job.

In [4]:
print(line)
for prcfil in prcfils:
    if os.path.exists(prcfil):
        dbr = desc.wfmon.MonDbReader(prcfil, fix=False)
        dbr.tables(lev)
        print(dbr.table('task').query('task_id<10').task_stderr)
    else:
        print(f"File not found: {sysfil}")
    print(line)

----------------------------------------------------------------------------
DB runinfo/monitoring.db has 7 tables
*******************************************************
Table workflow has 1 rows and 10 columns
Column names:
    object   run_id
    object   workflow_name
    object   workflow_version
    object   time_began
    object   time_completed
    object   host
    object   user
    object   rundir
     int64   tasks_failed_count
     int64   tasks_completed_count
*******************************************************
Table task has 80000 rows and 15 columns
Column names:
     int64   task_id
    object   run_id
    object   task_depends
    object   task_func_name
    object   task_memoize
    object   task_hashsum
    object   task_inputs
    object   task_outputs
    object   task_stdin
    object   task_stdout
    object   task_stderr
    object   task_time_invoked
    object   task_time_returned
     int64   task_fail_count
   float64   task_fail_cost
*******************

In [28]:
print(dbr['workflow']['time_began'])
print('--------------------------------------')
print(dbr['try'][['task_try_time_launched','task_try_time_running','task_try_time_returned']])
print('--------------------------------------')
t0s = float(sym['time'][0])
print(t0s)
st0s = desc.wfmon.MonDbReader.string_from_time(t0s)
print(st0s)

0    2022-10-21 11:34:57.666452
Name: time_began, dtype: object
--------------------------------------
           task_try_time_launched       task_try_time_running  \
0      2022-10-21 11:35:12.803158  2022-10-21 11:35:14.536542   
1      2022-10-21 11:35:12.816540  2022-10-21 11:35:14.528150   
2      2022-10-21 11:35:12.818264  2022-10-21 11:35:14.536525   
3      2022-10-21 11:35:12.820011  2022-10-21 11:35:14.532770   
4      2022-10-21 11:35:12.821672  2022-10-21 11:35:14.503057   
...                           ...                         ...   
79995  2022-10-21 11:37:42.945268                        None   
79996  2022-10-21 11:37:42.946757                        None   
79997  2022-10-21 11:37:42.948583                        None   
79998  2022-10-21 11:37:42.950203                        None   
79999  2022-10-21 11:37:42.952718                        None   

           task_try_time_returned  
0      2022-10-21 11:35:36.765025  
1      2022-10-21 11:35:29.536823  
2      2

In [27]:
dbr.fix_tasks()
print(f"Run IDs: {dbr.run_ids}")
print(f"Task names: {dbr.task_names}")
print(f"Task counts: {dbr.task_name_counts}")
pfm = desc.wfmon.PerfStatLogReader(4)
nskip = 0
nkeep = 0
tsk = dbr.table('task')
print(tsk['task_id'])
for tid in range(len(tsk)):
    fnam = dbr.task_logs[tid]
    indict = collections.OrderedDict()
    for nam in ['run_idx', 'task_idx', 'task_id']: indict[nam] = tsk[nam][tid]
    if os.path.exists(fnam):
        nkeep = nkeep + 1
        #print(f"  {fnam}")
        pfm.read(fnam, indict)
    else:
        nskip = nskip + 1
print(f"nskip = {nskip}")
print(f"nkeep = {nkeep}")
print(f"nfile = {pfm.nfile}")
print(f" nval = {pfm.nval}")
print(f"nmiss = {pfm.nmissing}")
print(f"Perf state names: {pfm.dict.keys()}")
print(pfm.table())
#print(pfm.dict)

Run IDs: ['464a23dc-50b0-4f8e-bbfe-bd99da1aea6d']
Task names: ['mybash_ifix']
Task counts: [80000]
0            0
1            1
2            2
3            3
4            4
         ...  
79995    79995
79996    79996
79997    79997
79998    79998
79999    79999
Name: task_id, Length: 80000, dtype: int64


AttributeError: 'PerfStatLogReader' object has no attribute 'read'

In [ ]:
tsk = dbr.table('task')
print(tsk